# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulah-naeem/FlyRank-ml-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook establishes the **Data Contract** for the FlyRank ML Search Intelligence pipeline. Every claim about data grain, time windows, feature classifications, and availability is explicitly verified with executable DuckDB/Pandas queries.

> **Skill loaded:** `writing-data-contracts` + `flyrank/flyrank-data`

## 1. Plain-Words Data Contract (5 Core Answers)

1. **Unit of Analysis (Grain):** One row represents one pseudonymized content item (`content_id`) for a specific client (`client_id`).
2. **Table(s) Used:** `fact_content_daily_performance` (Warehouse Parquet release) and `content_refresh_anonymized.csv` (local starter dataset).
3. **Time Window:** Trailing 90-day aggregated window evaluated on a mid-panel month partition (`month=2026-03`).
4. **Target / Label to Predict:** Binary traffic decline indicator (`is_declining_label`, where `trend_direction == 'down'`). Predicts whether organic search traffic is dropping.
5. **Deliberately Excluded Field:** `trend_direction` and `trend_pct` — these fields are used directly to construct `is_declining_label`. Including them in features causes **100% Target Leakage** (injecting future label information into past predictions).

In [4]:
import os, sys, subprocess
import pandas as pd
import numpy as np
import duckdb

# Determine workspace root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

csv_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(csv_path):
    csv_path = "../../data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)

con = duckdb.connect()
con.register("raw_content", df)

print(f"Dataset Loaded Successfully: {len(df):,} rows | {df.shape[1]} columns")
print(f"Overall Declining Page Rate: {df['is_declining_label'].mean():.3f}")

Dataset Loaded Successfully: 30,000 rows | 45 columns
Overall Declining Page Rate: 0.542


## 2. Fields Classification (Feature / Label / Context / Excluded)

Every field in the dataset is classified into one of four buckets:

- **Features (Pre-decision signals):** `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `days_since_last_update`, `word_count`, `search_volume`, `cpc`, `has_gsc_data`.
- **Label (Target Variable):** `is_declining_label` (Binary 1/0 indicator derived from trailing trend).
- **Context (Grouping & Joining):** `content_id`, `client_id`, `content_type`, `main_intent` (Used for grouping & grouped train/test splitting, NEVER as direct model inputs).
- **Excluded (Leakage & Privacy):** `trend_direction` (Direct label leak), `trend_pct` (Direct label leak), `internal_client_name` (Privacy pseudonymization), `ai_traffic_pct` (Unstable cross-system measurement).

In [5]:
# Summary Table of Field Classification
field_buckets = pd.DataFrame([
    {"Field": "days_since_last_update", "Bucket": "Feature", "Role": "Content freshness metric"},
    {"Field": "impressions_90d", "Bucket": "Feature", "Role": "Historical search visibility"},
    {"Field": "ctr", "Bucket": "Feature", "Role": "Historical engagement rate (*100)"},
    {"Field": "avg_position", "Bucket": "Feature", "Role": "Google Search Console rank (0 = missing)"},
    {"Field": "word_count", "Bucket": "Feature", "Role": "Content length indicator"},
    {"Field": "is_declining_label", "Bucket": "Label", "Role": "Target variable (1 if trend_direction=='down')"},
    {"Field": "content_id", "Bucket": "Context", "Role": "Unique row identifier"},
    {"Field": "client_id", "Bucket": "Context", "Role": "Client grouping ID for split validation"},
    {"Field": "trend_direction", "Bucket": "Excluded", "Role": "TARGET LEAKAGE (used to derive label)"},
    {"Field": "trend_pct", "Bucket": "Excluded", "Role": "TARGET LEAKAGE (used to derive label)"}
])
display(field_buckets)

,Field,Bucket,Role
0,days_since_last_update,Feature,Content freshness metric
1,impressions_90d,Feature,Historical search visibility
2,ctr,Feature,Historical engagement rate (*100)
3,avg_position,Feature,Google Search Console rank (0 = missing)
4,word_count,Feature,Content length indicator
5,is_declining_label,Label,Target variable (1 if trend_direction=='down')
6,content_id,Context,Unique row identifier
7,client_id,Context,Client grouping ID for split validation
8,trend_direction,Excluded,TARGET LEAKAGE (used to derive label)
9,trend_pct,Excluded,TARGET LEAKAGE (used to derive label)


## 3. Verification Queries (Grain, Scope & Availability Checks)

We execute three mandatory verification queries using DuckDB:

1. **Query 1 (Grain Check):** Prove that `content_id` is unique per row (`HAVING COUNT(*) > 1` returns 0 rows).
2. **Query 2 (Scope & Row Count):** Measure total rows, unique clients, and content items.
3. **Query 3 (Availability Check with `IS TRUE`):** Measure valid data availability for GSC search rank and word count using `IS TRUE` filters.

In [7]:
# Query 1: Grain Check (Should return 0 rows)
grain_check = con.execute("""
    SELECT content_id, COUNT(*) as duplicate_count 
    FROM raw_content 
    GROUP BY content_id 
    HAVING duplicate_count > 1 
    LIMIT 5
""").df()

print("--- Query 1: Grain Check ---")
print(f"Duplicate rows found at grain (content_id): {len(grain_check)}")
assert len(grain_check) == 0, "Grain violation detected!"
print("Verdict: Grain holds perfectly (1 row = 1 content item).\n")

# Query 2: Scope and Counts
counts_df = con.execute("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT client_id) as total_clients,
        COUNT(DISTINCT content_id) as total_content_items
    FROM raw_content
""").df()

print("--- Query 2: Scope & Row Counts ---")
display(counts_df)

# Query 3: Availability Filter Check with IS TRUE logic
avail_df = con.execute("""
    SELECT 
        COUNT(*) as total_rows,
        SUM(CASE WHEN (avg_position > 0) IS TRUE THEN 1 ELSE 0 END) as gsc_data_available_rows,
        ROUND(AVG(CASE WHEN (avg_position > 0) IS TRUE THEN 1.0 ELSE 0.0 END) * 100, 2) as gsc_available_pct,
        SUM(CASE WHEN (word_count > 0) IS TRUE THEN 1 ELSE 0 END) as word_count_available_rows,
        ROUND(AVG(CASE WHEN (word_count > 0) IS TRUE THEN 1.0 ELSE 0.0 END) * 100, 2) as word_count_available_pct
    FROM raw_content
""").df()

print("--- Query 3: Data Availability Check (IS TRUE) ---")
display(avail_df)

--- Query 1: Grain Check ---
Duplicate rows found at grain (content_id): 0
Verdict: Grain holds perfectly (1 row = 1 content item).

--- Query 2: Scope & Row Counts ---


,total_rows,total_clients,total_content_items
0,30000,32,30000


--- Query 3: Data Availability Check (IS TRUE) ---


,total_rows,gsc_data_available_rows,gsc_available_pct,word_count_available_rows,word_count_available_pct
0,30000,28795.0,95.98,22301.0,74.34


## 4. Five-Feature Frame & Deliberate Leakage Trap Experiment

### The 5 Honest Features (Max 5)
1. `days_since_last_update`: *Knowable at decision moment because content publish/update timestamp is recorded in CMS prior to evaluation date.*
2. `impressions_90d`: *Knowable at decision moment because search impressions are accumulated over historical 90-day period up to evaluation date.*
3. `ctr`: *Knowable at decision moment because click-through rate is logged historically before prediction moment.*
4. `avg_position_clean`: *Knowable at decision moment because GSC search rank is recorded prior to evaluation time.*
5. `has_gsc_data`: *Knowable at decision moment because system tracking flag indicates whether GSC search data exists prior to model scoring.*

### The Trap (Deliberate Leakage Experiment)
We deliberately add `trend_pct` (a label-derived column) into our feature vector, observe the artificial "perfect" precision score, and then purge it to recover the true, honest performance metric.

In [8]:
from sklearn.tree import DecisionTreeClassifier

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return float(topk.mean())

# Build clean 5-feature vector
df["avg_position_clean"] = np.where(df["avg_position"] > 0, df["avg_position"], np.nan)
df["has_gsc_data"] = (df["avg_position"] > 0).astype(int)

honest_features = ["days_since_last_update", "impressions_90d", "ctr", "has_gsc_data", "word_count"]
X_honest = df[honest_features].fillna(0)
y = df["is_declining_label"].values

# Fit honest decision tree model
dt_honest = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_honest.fit(X_honest, y)
honest_scores = dt_honest.predict_proba(X_honest)[:, 1]
honest_p50 = precision_at_k(honest_scores, y, k=50)

print(f"--- 1. Honest 5-Feature Model ---")
print(f"Features: {honest_features}")
print(f"Honest Model Precision@50: {honest_p50:.3f}\n")

# --- THE TRAP: Deliberately Inject Label-Derived Column (trend_pct) ---
leaky_features = honest_features + ["trend_pct"]
X_leaky = df[leaky_features].fillna(0)

dt_leaky = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_leaky.fit(X_leaky, y)
leaky_scores = dt_leaky.predict_proba(X_leaky)[:, 1]
leaky_p50 = precision_at_k(leaky_scores, y, k=50)

print(f"--- 2. Deliberate Leakage Trap Experiment ---")
print(f"Added Leaky Feature: 'trend_pct'")
print(f"TRAP Model Precision@50: {leaky_p50:.3f} (ARTIFICIAL PERFECT SCORE!)")
print("Explanation: 'trend_pct' was used to derive 'is_declining_label'. Feeding it to the model leaks the answer directly!\n")

# --- PURGE THE TRAP ---
print("--- 3. Purging the Trap ---")
print("Target-derived column 'trend_pct' REMOVED from feature vector.")
print(f"Restored Honest Model Precision@50: {honest_p50:.3f}")

--- 1. Honest 5-Feature Model ---
Features: ['days_since_last_update', 'impressions_90d', 'ctr', 'has_gsc_data', 'word_count']
Honest Model Precision@50: 0.720

--- 2. Deliberate Leakage Trap Experiment ---
Added Leaky Feature: 'trend_pct'
TRAP Model Precision@50: 1.000 (ARTIFICIAL PERFECT SCORE!)
Explanation: 'trend_pct' was used to derive 'is_declining_label'. Feeding it to the model leaks the answer directly!

--- 3. Purging the Trap ---
Target-derived column 'trend_pct' REMOVED from feature vector.
Restored Honest Model Precision@50: 0.720


## 5. Data Limits & Named Slice Limitations

**Named Limitation of this Data Slice:**
1. **Missingness Follows Category:** `avg_position = 0` does NOT mean rank zero; it indicates no search data recorded in GSC (~25.7% of pages).
2. **Unbalanced Client History Depth:** Clients enter the system at different dates (`gsc_data_start`). Rows prior to client onboarding (`ga4_data_available = FALSE`) have engagement metrics zero-filled.
3. **Blind Fillna Risk:** Blindly filling missing values with 0 injects a category signal. We handle missingness by introducing explicit boolean availability flags (`has_gsc_data`).

In [9]:
# Code verification of named data limitations
missing_summary = con.execute("""
    SELECT 
        content_type,
        COUNT(*) as total_items,
        SUM(CASE WHEN avg_position = 0 THEN 1 ELSE 0 END) as gsc_missing_count,
        ROUND(AVG(CASE WHEN avg_position = 0 THEN 1.0 ELSE 0.0 END) * 100, 1) as gsc_missing_pct
    FROM raw_content
    GROUP BY content_type
    ORDER BY total_items DESC
""").df()

print("--- Named Limitation Verification: Missingness by Content Type ---")
display(missing_summary)

--- Named Limitation Verification: Missingness by Content Type ---


,content_type,total_items,gsc_missing_count,gsc_missing_pct
0,keyword article,27207,475.0,1.7
1,feedly article,2096,730.0,34.8
2,comparison article,697,0.0,0.0
